<a href="https://colab.research.google.com/github/alexarnoldy/AI-ML/blob/master/chapter01/chapter01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Chapter 1 - Introducing Agents</h1>
<i>Exploring AI Agents</i>


<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents/blob/main/chapter01/chapter01.ipynb)

---

This notebook is for Chapter 1 of [An Illustrated Guide to AI Agents](https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ) by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [1]:
 %%capture
 !pip install illustrated-agents


<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Building an `Agent` from Scratch

Throughout this book, we will explore various components of Agent, which together make up the `TinyAgent` which is given in the `\tinyagent` folder.
This code builds an Agent (called `TinyAgent`) entirely from scratch, using only API calls to the LLM.

To do so, we make use of a highly modular and educational structure of our `TinyAgent` that focuses on understanding the vital components of an Agent.
As such, we start this chapter with an `agent.py` and slowly build it up by adding components:

![../images/overview.png](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents/blob/main/images/overview.png?raw=1)

Each chapter builds on the previous one, starting with this one which creates the `agent.py` file. In each chapter, we add a module to the `agent.py` framework so that we can improve its behavior. We will be iteratively updating `agent.py` to implement this behavior:

| Chapter | Module Added | Capability |
|---------|--------------|------------|
| 1 | (skeleton) | Basic structure with placeholders |
| 2 | `llm.py` | Can query an LLM (stateless) |
| 3 | - | Learn to do reasoning (both native and prompted) |
| 4 | `memory.py` | Remembers conversation history |
| 5 | `tools.py` & `toolbox.py` | Toolcalling capabilities (both native and prompted) |
| 6 | `planning.py` | ReAct loop (multi-step reasoning) |
| 8 | `coordinator.py` | Multi-agent collaboration |

LLMs these days have been trained to perform native tool calling and reasoning. Although this makes it easier to showcase these capabilities, it is a bit of a black box to understand **how** they are reasoning and performing tool calls. For that reason, we thought it would be interesting in this book to show how to use a model that cannot reason or call tools natively and have it do that through prompt engineering. We hope that it will help you understand what it actually means for a model to call a tool or to reason. However, that doesn't mean we can forget the models that **can** perform native reasoning and tool calling.

Therefore, we decided to show you both! Throughout the book we will demonstrate both prompt-based and native reasoning and tool calling capabilities. By doing it yourself, we hope it will help you get insight into how these newer models have learned to perform native reasoning and tool calling.

As such, we choose two models throughout this book, one that cannot natively perform any reasoning or tool calling whatsoever (Gemma 3) and one that has been trained specifically to perform agentic tasks through reasoning and tool calling (Gemma 4).

![../images/evolution.png](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents/blob/main/images/evolution.png?raw=1)

We strongly believe that having a model like Gemma 3 perform tool calling is a wonderful way to understand what is needed in the process of calling a tool. It will answer questions like; What format do we need to call a tool? Why is XML better than JSON for multi-line texts in tool calls?

By getting your hands dirty, you will learn more about the complexities of these capabilities than using a model that does it all for you! As such, we will start with Gemma 3 and slowly built up to Gemma 4. We will demonstrate both prompt-based and native capabilities and slowly built up so that you will get a thorough understanding of these techniques.

What you will learn with these two models should apply to other open source and proprietary models, and we've included code so you can tweak and try other models as they are released in the future.

## 2 - The `TinyAgent`

Let's start with building our `TinyAgent`. This class will start with the LLM we defined above and nothing more. It technically cannot be considered an Agent yet until we have added all above components.

![../images/ch1.png](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents/blob/main/images/ch1.png?raw=1)


That said, let's build the scaffolding of this `TinyAgent`. There are several components that we can already define without using any LLM, namely:

* Attributes that we add later on
* `run` which is the main function to run the Agent
* `_step` which is a single step of the Agent (it may run for multiple steps)

In [2]:
class TinyAgent:
    """A minimal, modular, and educational agent framework."""

    def __init__(self):
        self.llm = None  # Chapter 2 & 3: Add LLM
        self.memory = None  # Chapter 4: Add Memory
        self.tools = None  # Chapter 5: Add Tools
        self.planner = None  # Chapter 6: Add Planning

    def run(self, task: str) -> str:
        """Run the agent on a task."""
        return self._step(task)

    def _step(self, task: str) -> str:
        """Perform a single step."""
        # Placeholder - will be implemented in later chapters
        return f"Received: {task}"

    def _execute_action(self, action: str) -> str:
        """Execute a tool action."""
        # Placeholder - will be implemented in later chapters
        return f"Executed action: {action}"

Looking a little bit closer at each function gives us more clarity with respect to the difference between the `run` and a single `_step`:

In [3]:
from illustrated_agents.chapters.ch1 import agent_annotated; agent_annotated

Then, all that is left to do is "run" the Agent, which gives back nothing more than what is has received. After all, without a "brain", there is nothing more than your `TinyAgent` can do:

In [4]:
agent = TinyAgent()
agent.run("What is 2 + 2?")

'Received: What is 2 + 2?'

So... let's continue to the next chapter where we can give it a "brain"!

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

We will end each chapter by giving you an overview of what we built. Building your `TinyAgent` from scratch and through modularity means that we can continuously track which pieces of code we have updated and how.

All code created in this chapter tutorials can be seen in `src/illustrated_agents` where you will find an overview like this:

  ```
  TinyAgent
  ├── __init__.py
  ├── agent.py
  ├── cli.py
  ├── display.py
  ├── llm.py
  ├── memory.py
  ├── planning.py
  ├── skills.py
  ├── toolbox.py   
  ├── tools.py
  ├── trajectory.py
  └── utils.py
  ```

The idea is that we slowly built up each of those `.py` files and explain how they relate to Agents!

In this chapter, we covered the basic structure of your `TinyAgent`, which means that you started taking your first steps into building an agent from scratch, congratulations :)

Run the following for an overview:

In [5]:
from illustrated_agents.chapters.ch1 import what_we_built; what_we_built

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ └── agent.py ← New (`TinyAgent` skeleton - The start of something cool ;))                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# What's Next

In the next chapter, we cover how LLMs work, which inference providers you can choose from, and how we are going to use it as the "brain" in your `TinyAgent`.